## 📂 Prodigy Task 3 — Decision Tree Classifier
---
We are given bank - marketing data that contains demographic and behavioural infomation about account holders. Out goal is to build a decision tree classifier that will predict whether an account holder will opt for a scheme or not. The dataset is slighly larger than the previously shared one.

---

#### **Step 1:-** $\textbf{🗂️ Import the Necessary Libraries}$

In [1]:
import pandas as pd

In [2]:
import numpy as np

In [3]:
import sklearn as sk

In [4]:
import matplotlib.pyplot as plt

In [5]:
bank = pd.read_csv("bank-full.csv", sep=";")
bank.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


#### **Step 2:-** $\textbf{Explore the data 🌅}$

We will be extracting dimension, column names and missing values in this step. Also we will get the data type of each column and the total number of unique values per column.

In [6]:
dim = bank.shape      # Dimension of the dataset
print("="*50)
print(f"This dataset has {dim[0]} rows and {dim[1]} columns")
print("="*50)

This dataset has 45211 rows and 17 columns


In [7]:
name = bank.columns    # Column names
print("The following are the column names:-")
print(name)

The following are the column names:-
Index(['age', 'job', 'marital', 'education', 'default', 'balance', 'housing',
       'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays',
       'previous', 'poutcome', 'y'],
      dtype='str')


In [8]:
bank.isna().sum()      # Missing values per column

age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
day          0
month        0
duration     0
campaign     0
pdays        0
previous     0
poutcome     0
y            0
dtype: int64

This dataset has no missing values.

In [9]:
bank.dtypes         # Data type of each column

age          int64
job            str
marital        str
education      str
default        str
balance      int64
housing        str
loan           str
contact        str
day          int64
month          str
duration     int64
campaign     int64
pdays        int64
previous     int64
poutcome       str
y              str
dtype: object

In [10]:
bank.nunique()        # total number of unique values per column.

age            77
job            12
marital         3
education       4
default         2
balance      7168
housing         2
loan            2
contact         3
day            31
month          12
duration     1573
campaign       48
pdays         559
previous       41
poutcome        4
y               2
dtype: int64

#### **Step 3:-** $\textbf{Make list of column names}$
Our aim to assign numerical values (class) to each of the categorical data. But there are certain columns that have lots of distinct classes. So we are going to use different types of encoders depending upon the total number of unique values. It unique value < 5 we will use onehotencoder else we will use ordinalencoder.

In [31]:
low_cl = ["marital", "education", "default", "housing", "loan", "contact", "poutcome"]
high_cl = ["job", "month"]
int_cl = ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']

#### **Step 4:-** $\textbf{Encode the Listed Columns}$

In this step we are going to encode the column depending upon the unique value count per column. Later we will combine all the new columns togther to form a design matrix $X$ and a response column $Y$

In [32]:
e1 = sk.preprocessing.OneHotEncoder(sparse_output = False, drop = 'first')
e2 = sk.preprocessing.OrdinalEncoder()

x1 = e1.fit_transform(bank[low_cl])
x2 = e2.fit_transform(bank[high_cl])
x3 = bank[int_cl].values
X = np.column_stack([x1,x2,x3])
Y = np.where(bank['y']=='no', 0, 1)

#### **Step 5:-** $\textbf{🚂 Perform Train Test Split}$

We randomly select 80% of the data for training and remaining 20% for testing the model.

In [33]:
x_train, x_test, y_train, y_test = sk.model_selection.train_test_split(X,Y, test_size = 0.2, random_state = 121)

#### **Step 6:-** $\textbf{🤺 Train the Model}$
We will use the training data to train the model. Later we use the test data to assess it quality by calculating some standard metrics.

In [34]:
model = sk.tree.DecisionTreeClassifier()
model.fit(x_train, y_train)
y_hat = model.predict(x_test)               # Predcted outout of the test data
tn, fp, fn, tp = sk.metrics.confusion_matrix(y_test, y_hat).ravel()
accuracy = (tn + tp)/(tn + fp + fn + tp)
sen = tp/(tp + fn)        # Sensiitvity
pre = tp/(tp + fp)        # Precision
speci = tn/(tn+ fp)       # Specificity

#### **Step 7:-** $\textbf{Repeat Step 5 and 6 }$
We are going to train the model 2000 times and collect the respective metrics in the separate list. We will take the average of all these values so that we can be confident regarding the randomness of these metrics as they are subject to sample variation.

In [49]:
rep = 2000
acc = []
sens = []
speci = []
pre = []

for i in range(rep):
    x_train, x_test, y_train, y_test = sk.model_selection.train_test_split(X,Y, test_size = 0.2, random_state = i)
    model = sk.tree.DecisionTreeClassifier(class_weight = 'balanced', max_depth = 5, min_samples_leaf = 20)
    model.fit(x_train, y_train)
    y_hat = model.predict(x_test)               # Predcted outout of the test data
    tn, fp, fn, tp = sk.metrics.confusion_matrix(y_test, y_hat).ravel()
    acc_temp = (tn + tp)/(tn + fp + fn + tp)
    sen_temp = tp/(tp + fn)        # Sensiitvity
    pre_temp = tp/(tp + fp)        # Precision
    speci_temp = tn/(tn+ fp)       # Specificity

    # Store the values
    acc.append(acc_temp)
    sens.append(sens_temp)
    speci.append(speci_temp)
    pre.append(pre_temp)

In [50]:
print(f"Average accuracy of the model:- {np.mean(acc)}")
print(f"Average sensitivity of the model:- {np.mean(sens)}")
print(f"Average specificity of the model:- {np.mean(speci)}")
print(f"Average precision of the model:- {np.mean(pre)}")

Average accuracy of the model:- 0.809487780603782
Average sensitivity of the model:- 0.8665464382326422
Average specificity of the model:- 0.8105512461932101
Average precision of the model:- 0.35973305961561075


**Interpretation of the above metrics:-**
1. There has been a trade off between overall accuracy and sensitivity of the model. Out of all the actual customers that said *yes* for the scheme, our model has identified 86% of them.
2. Similarly out of all the actual customers that said *no* to the scheme, our model has identified almost 81% of them.
3. However the model has very poor precision score. This means that out of all the predicted person that would likely agree to the scheme, only 35% of them actually said *yes*.
4. The model is good at identifying the people that will not agree to the scheme. Quite contrast to what we actually wanted. 